In [11]:
import pybullet as pb
import numpy as np
import time
from selfbalancebot import SelfBalanceSim
from backstepnn_sim import BacksteppingNN
import matplotlib.pyplot as plt

class SelfBalanceExp(SelfBalanceSim):
    def __init__(self, del_t=1 / 240, runtime = 20, L = 10, gam_w = 30, gam_v = 30,k1=5, k2=5, sig_w = 0.1, sig_v = 0.1):
        self.Controller = BacksteppingNN(2, L, 1)     
        self.runtime = runtime
        self.k1 = k1
        self.k2 = k2
        self.L = L
        self.gam_w = gam_w
        self.gam_v = gam_v
        self.sig_w = sig_w
        self.sig_v = sig_v


        super().__init__(del_t)
        

    def Start(self, start_pos=[0,0,0.2], start_orientation=pb.getQuaternionFromEuler([0.0,0.0,0.0])):
        super().Start(start_pos, start_orientation) 
        self.start_time = time.time()        
        self.tspan = []
        self.x_history = []
        self.xref_history = []
        self.e_history = []
        self.u_history = []
        self.Wfn_history = []
        self.Vfn_history = []

    def g(self, x):
        """
        Calculates the g4 component of the control vector g(x) for the self_balance_bot.
        This represents the mapping from total motor torque to angular acceleration (theta_ddot).
        
        Parameters:
        x3 (float): The tilt angle (theta) in radians.
        
        Returns:
        float: The value of the g4 term.
        """
        # Physical constants derived from URDF
        Mt = 1.75185    # Total effective inertial mass
        It = 0.09381    # Total body inertia about the axle
        Ml = 0.23648    # Mass * distance to center of mass
        r = 0.20006     # Wheel radius
        
        # Pre-calculate trigonometric values
        cos_x3 = np.cos(x)
        
        # Expanded Numerator: -Mt - (Ml/r)*cos(x3)
        numerator = -Mt - (Ml / r) * cos_x3
        
        # Expanded Denominator (Determinant D): Mt*It - (Ml*cos(x3))^2
        denominator = (Mt * It) - (Ml * cos_x3)**2
        
        return numerator / denominator

        
    def Update(self):
        self.get_states()
        _,_,x1,x2 = self.states
        x = np.array([x1, x2])
        xref = 0
        dxref = 0
        ddxref = 0

        # Generate Input from controller
        input, e1, e2 = self.Controller.compute_control(x, xref, dxref, ddxref, self.k1, self.k2,self.g(x1)) 
        # Apply input
        input_limit = 4
        input_with_limit = min(input_limit, max(-input_limit, input[0]))
        print(input_with_limit)
        self.apply_input(input_with_limit, 'torque')

        # Record Data
        self.tspan.append(time.time()-self.start_time)
        self.x_history.append(x.copy())
        self.xref_history.append(xref)
        self.e_history.append(np.array([e1, e2]))
        self.u_history.append(input_with_limit)
        self.Wfn_history.append(np.linalg.norm(self.Controller.W,'fro'))
        self.Vfn_history.append(np.linalg.norm(self.Controller.V,'fro'))

        self.Controller.update_weights(x, e1, e2, np.eye(self.L+1)*self.gam_w, np.eye(3)*self.gam_v, self.sig_w, self.sig_v, self.dt)

        if (time.time()-self.start_time > self.runtime):
            self.running = False

        keys = pb.getKeyboardEvents()
        force = 50 # Additive force Magnitude
        if ord('f') in keys and keys[ord('f')] & pb.KEY_WAS_TRIGGERED:
            link_id = 0
            pb.applyExternalForce(
                objectUniqueId=self.model,
                linkIndex=link_id,   # -1 for base, otherwise link number
                forceObj=[0, force, 0], # force vector (Newton)
                posObj=[0, 0, 1],    # position in link frame or world
                flags=pb.LINK_FRAME  # or pb.WORLD_FRAME
            )
        
        if ord('d') in keys and keys[ord('d')] & pb.KEY_WAS_TRIGGERED:
            link_id = 0
            pb.applyExternalForce(
                objectUniqueId=self.model,
                linkIndex=link_id,   
                forceObj=[0, -force, 0], # force vector (Newton)
                posObj=[0, 0, 1],    # position in link frame or world
                flags=pb.LINK_FRAME  # or pb.WORLD_FRAME
            )

    def get_data(self):
        return self.tspan, self.x_history, self.e_history, self.u_history, self.Wfn_history, self.Vfn_history

    

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

def plot_tracking(tspan, x_history, xref_history, e_history, u_history):
    t = np.array(tspan)
    x = np.array(x_history)
    xref = np.array(xref_history)
    e = np.array(e_history)
    u = np.array(u_history)

    plt.figure(figsize=(8,10))

    # ---- Subplot 1: x1, x2, xref ----
    plt.subplot(3,1,1)
    plt.plot(t, x[:,0], label=r'$x_1$ (Tilt)')
    plt.plot(t, x[:,1], label=r'$x_2$(Tilt rate)')
    plt.plot(t, xref, '--', label=r'$x_{\mathrm{ref}}$ (Tilt ref)')
    plt.title(r'Tracking Performance')
    plt.xlabel(r'Time (s)')
    plt.ylabel(r'States')
    plt.legend()
    plt.grid()

    # ---- Subplot 2: error ----
    plt.subplot(3,1,2)
    plt.plot(t, e[:,0], label=r'$e_1$')
    plt.plot(t, e[:,1], label=r'$e_2$')
    plt.title(r'Tracking Error')
    plt.xlabel(r'Time (s)')
    plt.ylabel(r'Error')
    plt.legend()
    plt.grid()

    # ---- Subplot 3: control ----
    plt.subplot(3,1,3)
    plt.plot(t, u, label=r'$u$')
    plt.title(r'Control Input')
    plt.xlabel(r'Time (s)')
    plt.ylabel(r'Wheel Torque')
    plt.legend()
    plt.grid()

    plt.tight_layout()
    plt.show()

def plot_weight_norms(tspan, Wfn_history, Vfn_history):
    t = np.array(tspan)
    W = np.array(Wfn_history)
    V = np.array(Vfn_history)

    plt.figure(figsize=(8,5))

    plt.plot(t, W, label=r'$\|\hat{W}\|_F$')
    plt.plot(t, V, label=r'$\|\hat{V}\|_F$')

    plt.title(r'Weight Norms')
    plt.xlabel(r'Time (s)')
    plt.ylabel(r'Frobenius Norm')
    plt.legend()
    plt.grid()

    plt.tight_layout()
    plt.show()

%matplotlib qt
f = SelfBalanceExp(runtime=300,L=10, gam_v=30, gam_w=30, sig_w = 0.1, sig_v = 0.1)
t, x, e, u, W, V = f.get_data()
plot_tracking(t, x,f.xref_history, e, u)
plot_weight_norms(t, W, V)


Started...
0.007003354072154278
0.006500867232718354
0.006002442488605109
0.005515449173042637
0.00503725314909104
0.004568657442677855
0.004110254048066806
0.003662526455233074
0.0032258909617481806
0.0028007313459566035
0.0023873411760649077
0.0019860131694872307
0.0015969991734876815
0.0012205144748959624
0.000856740265780297
0.0005058253486880242
0.00016788752247288817
-0.0001569851708107909
-0.00046873326349734215
-0.0007673247131731454
-0.0010527537732082902
-0.001325039877279464
-0.001584226535082902
-0.0018303812980434617
-0.0020635925827160347
-0.0022839691075705218
-0.0024916395792633364
-0.002686751515506664
-0.002869470151055641
-0.0030399773849160727
-0.003198470752923856
-0.003345162419791691
-0.003480278188541451
-0.003604056527237906
-0.0037167476132029024
-0.003818612395364678
-0.003909921363656696
-0.0039909540520525364
-0.004061998640143037
-0.004123199579016407
-0.004174790210143228
-0.00421749880821209
-0.004251530728644402
-0.004277142912048073
-0.0042956383644827